# Step 1: Test the connection

In [6]:
import ollama

answer = ollama.chat(
    model = "qwen2.5:14b",
    messages = [
        {
            "role": "user",
            "content": "What is the capital of Norway?" # Say hello in one word
        }
    ]
)

In [7]:
print(type(answer).__name__)

ChatResponse


In [8]:
answer

ChatResponse(model='qwen2.5:14b', created_at='2026-10-07T01:15:19.8092199Z', done=True, done_reason='stop', total_duration=12174218000, load_duration=10966788200, prompt_eval_count=36, prompt_eval_duration=368787000, eval_count=8, eval_duration=832223000, message=Message(role='assistant', content='The capital of Norway is Oslo.', thinking=None, images=None, tool_name=None, tool_calls=None), logprobs=None)

In [9]:
answer.message

Message(role='assistant', content='The capital of Norway is Oslo.', thinking=None, images=None, tool_name=None, tool_calls=None)

In [10]:
answer.message.content

'The capital of Norway is Oslo.'

# Step 2: The model does not remember

In [11]:
a1 = ollama.chat(model="qwen2.5:14b", messages=[{"role": "user", "content": "My name is Caleb."}])
a2 = ollama.chat(model="qwen2.5:14b", messages=[{"role": "user", "content": "What's my name?"}])

print(a1.message.content)
print(a2.message.content)

Hello Caleb! It's nice to meet you. How can I assist you today?
As an artificial intelligence, I don't have access to personal information about the user, so I don't know your name. You can tell me your name if you'd like to!


# Step 3: **`call`:** use the chat as if it were a function

In [13]:
class Chat:
    def __init__(self, model = "qwen2.5:14b", system = "Answer in English, in a single sentence."):
        self.model = model
        self.system = system
        self._messages = []

    def __call__(self, text):
        self._messages.append({"role": "user", "content": text})
        answer = ollama.chat(
            model = self.model,
            messages = [{"role": "system", "content": self.system}, *self._messages],
        )
        content = answer.message.content
        self._messages.append({"role": "assistant", "content": content})
        return content

In [14]:
chat = Chat()
print(chat("Hi, my name is Caleb."))
print(chat("What's my name?"))

Hello Caleb, it's nice to meet you.
Your name is Caleb.


In [15]:
chat._messages

[{'role': 'user', 'content': 'Hi, my name is Caleb.'},
 {'role': 'assistant', 'content': "Hello Caleb, it's nice to meet you."},
 {'role': 'user', 'content': "What's my name?"},
 {'role': 'assistant', 'content': 'Your name is Caleb.'}]

# Step 4: **`repr`** and **`str`:** how the chat is displayed

In [23]:
class Chat:
    def __init__(self, model="qwen2.5:14b", system = "Answer in English, in a single sentence."):
        self.model = model
        self.system = system
        self._messages = []

    def __call__(self, text):
        self._messages.append({"role": "user", "content": text})
        answer = ollama.chat(
            model = self.model,
            messages = [{"role": "system", "content": self.system}, *self._messages],
        )
        content = answer.message.content
        self._messages.append({"role": "assistant", "content": content})
        return content

    def __repr__(self):
        return f"Chat(model={self.model!r}, messages = {len(self._messages)})"

    def __str__(self):
        labels = {"user": "you", "assistant": self.model}
        return "\n".join(f"{labels[m['role']]}: {m['content']}" for m in self._messages)

In [24]:
chat = Chat()
print(chat("Hi, my name is Caleb."))
print(chat("What's my name?"))

Hello Caleb, nice to meet you.
Your name is Caleb.


In [26]:
print(repr(chat))
print("="*20)
print(chat)

Chat(model='qwen2.5:14b', messages = 4)
you: Hi, my name is Caleb.
qwen2.5:14b: Hello Caleb, nice to meet you.
you: What's my name?
qwen2.5:14b: Your name is Caleb.


# Step 5: The chat as a collection: **`len`, `getitem`, `contains`**

In [39]:
class Chat:
    def __init__(self, model="qwen2.5:14b", system = "Answer in English, in a single sentence."):
        self.model = model
        self.system = system
        self._messages = []

    def __call__(self, text):
        self._messages.append({"role": "user", "content": text})
        answer = ollama.chat(
            model = self.model,
            messages = [{"role": "system", "content": self.system}, *self._messages],
        )
        content = answer.message.content
        self._messages.append({"role": "assistant", "content": content})
        return content

    def __repr__(self):
        return f"Chat(model={self.model!r}, messages = {len(self._messages)})"

    def __str__(self):
        labels = {"user": "you", "assistant": self.model}
        return "\n".join(f"{labels[m['role']]}: {m['content']}" for m in self._messages)

    def __len__(self):
        return len(self._messages)

    def __getitem__(self, index):
        return self._messages[index]

    def __contains__(self, item):
        return any(item.lower() in m["content"].lower() for m in self._messages)

In [40]:
chat = Chat()
print(chat("What is `__len__` used for?"))
print(chat("And `__call__`?"))

The `__len__` method in Python is used to return the length of the object, which is called when the built-in `len()` function is used on the object.
The `__call__` method in Python allows a class's instance to be called as a function, enabling the instance to be invoked like a regular function with parentheses.


In [41]:
print(len(chat))

4


In [42]:
print(chat)

you: What is `__len__` used for?
qwen2.5:14b: The `__len__` method in Python is used to return the length of the object, which is called when the built-in `len()` function is used on the object.
you: And `__call__`?
qwen2.5:14b: The `__call__` method in Python allows a class's instance to be called as a function, enabling the instance to be invoked like a regular function with parentheses.


In [43]:
chat[1]

{'role': 'assistant',
 'content': 'The `__len__` method in Python is used to return the length of the object, which is called when the built-in `len()` function is used on the object.'}

In [44]:
print("__LEN__" in chat)

True


# Step 6: **`enter`** / **`exit`**: The chat as a context manager

In [48]:
class Chat:
    def __init__(self, model="qwen2.5:14b", system = "Answer in English, in a single sentence."):
        self.model = model
        self.system = system
        self._messages = []

    def __call__(self, text):
        self._messages.append({"role": "user", "content": text})
        answer = ollama.chat(
            model = self.model,
            messages = [{"role": "system", "content": self.system}, *self._messages],
        )
        content = answer.message.content
        self._messages.append({"role": "assistant", "content": content})
        return content

    def __repr__(self):
        return f"Chat(model={self.model!r}, messages = {len(self._messages)})"

    def __str__(self):
        labels = {"user": "you", "assistant": self.model}
        return "\n".join(f"{labels[m['role']]}: {m['content']}" for m in self._messages)

    def __len__(self):
        return len(self._messages)

    def __getitem__(self, index):
        return self._messages[index]

    def __contains__(self, item):
        return any(item.lower() in m["content"].lower() for m in self._messages)

    def __enter__(self):
        print(f"[open session with {self.model}]")
        # print("Entering the context manager...")
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        # print("Exiting the context manager...")
        with open("transcript.txt", "w", encoding= "utf-8") as file:
            file.write(str(self))
        print(f"[closed session: {len(self)} messages saved to transcript.txt]")
        return False  # Don't suppress exceptions

In [46]:
with Chat() as chat:
    print(chat("What's a context manager?"))
    print("message inside the with block:", len(chat))

print(repr(chat))

Entering the context manager...
A context manager is an object that defines the methods to set up and tear down context, typically used in a with statement for resource management in Python.
message inside the with block: 2
Exiting the context manager...
Chat(model='qwen2.5:14b', messages = 2)


In [49]:
with Chat() as chat:
    print(chat("What's a context manager?"))
    print("message inside the with block:", len(chat))

print(repr(chat))

[open session with qwen2.5:14b]
A context manager is a Python object that defines the methods __enter__() and __exit__(), which allow for safe management of resources such as file streams and network connections.
message inside the with block: 2
[closed session: 2 messages saved to transcript.txt]
Chat(model='qwen2.5:14b', messages = 2)
